# 1、使用@Tool装饰器定义工具

In [ ]:
from langchain_core.tools import tool


@tool
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  # 默认是函数的名称
print(f"args = {add_number.args}") # 
print(f"description = {add_number.description}") # 默认是函数的说明信息
print(f"return_direct = {add_number.return_direct}") # 未设置默认值是false

name = add_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = add_number(a: int, b: int) -> int - 计算两个整数的和
return_direct = False


In [9]:
from langchain_core.tools import tool


@tool(
    "add_two_number",  # 直接放字符串，不要写 name_or_callable=
    return_direct=True,
)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")
print(f"return_direct = {add_number.return_direct}")

name = add_two_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = add_two_number(a: int, b: int) -> int - 计算两个整数的和
return_direct = True


In [10]:
add_number.invoke({"a": 10, "b": 20})


30

In [12]:
from langchain_core.pydantic_v1 import (
    BaseModel,
    Field,
)
from langchain_core.tools import tool


class FieldInfo(BaseModel):
    a: int = Field(description="第一个整形参数")
    b: int = Field(description="第二个整形参数")


@tool(
    "add_two_number",  # 直接放字符串，不要写 name_or_callable=
    return_direct=True,
    args_schema=FieldInfo,
)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")
print(f"return_direct = {add_number.return_direct}")

name = add_two_number
args = {'a': {'title': 'A', 'description': '第一个整形参数', 'type': 'integer'}, 'b': {'title': 'B', 'description': '第二个整形参数', 'type': 'integer'}}
description = add_two_number(a: int, b: int) -> int - 计算两个整数的和
return_direct = True


# StructuredTool 的 from_function()使用

In [14]:
from langchain_core.tools import StructuredTool

# 声明一个函数
def search_google(query: str):
    """根据查询词在谷歌上搜索相关信息。""" # 💡强烈建议加上这段注释
    return "最后查询的结果"

# 定义一个工具
my_tool = StructuredTool.from_function(  # 注意这里是 from_function
    func=search_google,
)

print(f"Tool Name: {my_tool.name}")
print(f"Tool Description: {my_tool.description}")

Tool Name: search_google
Tool Description: search_google(query: str) - 根据查询词在谷歌上搜索相关信息。


In [15]:
my_tool.invoke({"query":"中美AI的现状"})

'最后查询的结果'

In [17]:
# 1. 导入必要的库
from langchain_core.tools import StructuredTool
# 🌟 规范避坑：LangChain 底层深度依赖 Pydantic，为了防止 V1 和 V2 版本冲突，
# 官方推荐统一从 langchain_core 内部导入 pydantic_v1
from langchain_core.pydantic_v1 import BaseModel, Field

# ==============================================================================
# 📝 阶段 1：定义工具的“入参说明书” (Pydantic Schema)
#
# 核心作用：大模型非常笨，你必须用极其严谨的格式告诉它，调用这个工具需要传什么参数。
# 命名规范：通常命名为 "工具名+Input"，取代你原本笼统的 "FieldInfo"。
# ==============================================================================
class SearchInput(BaseModel):
    # Field(description="...") 是给大模型看的！大模型会根据这段描述，
    # 决定从用户的提问中提取什么词填到 query 里。
    query: str = Field(description="要检索的关键词，必须是高度概括的名词或短语")

# ==============================================================================
# 🛠️ 阶段 2：编写纯 Python 业务逻辑函数
# ==============================================================================
def search_google(query: str) -> str:
    """根据查询词在谷歌上联网搜索最新信息。当用户询问实时新闻、天气或未知事实时使用。""" 
    # 💡 极度重要：上面的 """...""" 叫 Docstring（文档字符串）。
    # 这绝对不是写给程序员看的注释！这是大模型的“工具选择指南”！
    # 大模型会阅读这段话，来决定到底该不该使用这个工具。
    
    print(f"🌍 [后台执行] 正在偷偷联网搜索: {query}...")
    # 这里我们用假数据模拟真实的搜索返回
    return f"【谷歌搜索结果】：关于'{query}'，今天是个大晴天，适合学习 LangChain！"

# ==============================================================================
# 🤖 阶段 3：将普通函数“封装”成大模型能懂的终极工具
# ==============================================================================
my_tool = StructuredTool.from_function(
    
    # 1. 挂载真正的执行函数
    func=search_google,
    
    # 2. 挂载入参说明书 (极其重要)
    # 告诉大模型：“请严格按照 SearchInput 规定的格式，给我生成 JSON 参数！”
    args_schema=SearchInput,
    
    # 3. 拦截器开关
    # 默认 False：工具执行完后，把结果（天气晴朗）扔回给大模型，让大模型重新组织语言回答用户。
    # 设为 True：工具执行完后，一脚踢开大模型，把结果原封不动地直接打印给用户（适合返回图表、文件路径等不需要 AI 废话的场景）。
    return_direct=False,
    
    # 4. 手动覆盖工具名（可选）
    # 如果不写，默认使用函数名 "search_google"
    name="Google_Search_Tool",
    
    # 5. 手动覆盖工具描述（可选）
    # 如果不写，默认提取函数里的 """...""" 作为描述
    # description="这是覆盖后的描述"
)

# ==============================================================================
# 🔍 阶段 4：开箱验货
# ==============================================================================
print("=" * 50)
print(f"🔧 工具名称: {my_tool.name}")
print(f"📖 工具描述: {my_tool.description}")
print(f"📝 参数要求: {my_tool.args_schema.schema()}")
print("=" * 50)

# 我们可以手动模拟大模型，传个参数进去跑跑看
print("\n▶️ 模拟执行工具:")
result = my_tool.invoke({"query": "北京天气"})
print(f"\n✅ 最终返回给大模型的数据: \n{result}")

🔧 工具名称: Google_Search_Tool
📖 工具描述: Google_Search_Tool(query: str) -> str - 根据查询词在谷歌上联网搜索最新信息。当用户询问实时新闻、天气或未知事实时使用。
📝 参数要求: {'title': 'SearchInput', 'type': 'object', 'properties': {'query': {'title': 'Query', 'description': '要检索的关键词，必须是高度概括的名词或短语', 'type': 'string'}}, 'required': ['query']}

▶️ 模拟执行工具:
🌍 [后台执行] 正在偷偷联网搜索: 北京天气...

✅ 最终返回给大模型的数据: 
【谷歌搜索结果】：关于'北京天气'，今天是个大晴天，适合学习 LangChain！
